In [1]:
import pandas as pd
import numpy as np

In [ ]:
class SimpleBatteryStorage:
    def __init__(self, states:np.ndarray, actions:np.ndarray, prices:np.ndarray, start_state:int, end_state:int):
        self.states = states.reshape(1, -1)
        self.actions = actions.reshape(1, -1)
        self.prices = prices.reshape(-1, 1)
        self.start_state = start_state
        self.end_state = end_state
        
    def get_state_transitions(self):
        state_base = (np.ones(shape=(self.states.shape[-1], self.actions.shape[-1])) * self.states.reshape(-1,1)).reshape(-1, 1)
        actions = np.tile(self.actions.T,(state_base.shape[0]//self.actions.T.shape[0],1))
        next_state = state_base + actions
        timestep_state_transition = np.concat([state_base, actions, next_state], axis=1)

        actions_clmn = timestep_state_transition[:,1][..., np.newaxis]
        prices = self.prices.T
        costs = (prices * actions_clmn).T.reshape(-1,1) 
        
        tiling = [1]*timestep_state_transition.shape[-1]
        tiling[0] = costs.shape[0]//timestep_state_transition.shape[0]
        state_transition = np.concat([np.tile(timestep_state_transition, reps=tiling).reshape(-1, 3), costs], axis=1)
        discharge_violation_ids = np.argwhere(state_transition[:, 2] < np.min(self.states)).squeeze()
        charge_violation_ids = np.argwhere(state_transition[:, 2] > np.max(self.states)).squeeze()
        
        state_transition[discharge_violation_ids, 2] = np.min(self.states)
        state_transition[charge_violation_ids, 2] = np.max(self.states)
        
        state_transition[np.concat([charge_violation_ids, discharge_violation_ids]), -1] = np.inf
        
        time_ids = np.tile(np.array(range(len(self.prices))), reps=[state_transition.shape[0]//len(self.prices),1])
        
        state_transition_matrix = np.concatenate([time_ids.T.reshape(-1, 1), state_transition], axis=1)
        return state_transition_matrix
    
    
class SimpleBatteryStorageChargesRestricted:
    def __init__(self, states:np.ndarray, actions:np.ndarray, prices:np.ndarray, start_state:int, end_state:int, max_charges:int):
        self.states = states.reshape(1, -1)
        self.actions = actions.reshape(1, -1)
        self.prices = prices.reshape(-1, 1)
        self.start_state = start_state
        self.end_state = end_state
        self.max_charges = max_charges
        
    def get_state_transitions(self):
        
        state_base = (np.ones(shape=(self.states.shape[-1]*(self.max_charges+1), self.actions.shape[-1])) * (self.states.reshape(-1,1))).reshape(-1, 1)
        print(state_base)
        # actions = np.tile(self.actions.T,(state_base.shape[0]//self.actions.T.shape[0],1))
        # next_state = state_base + actions
        # timestep_state_transition = np.concat([state_base, actions, next_state], axis=1)

        # actions_clmn = timestep_state_transition[:,1][..., np.newaxis]
        # prices = self.prices.T
        # costs = (prices * actions_clmn).T.reshape(-1,1) 
        
        # tiling = [1]*timestep_state_transition.shape[-1]
        # tiling[0] = costs.shape[0]//timestep_state_transition.shape[0]
        # state_transition = np.concat([np.tile(timestep_state_transition, reps=tiling).reshape(-1, 3), costs], axis=1)
        # discharge_violation_ids = np.argwhere(state_transition[:, 2] < np.min(self.states)).squeeze()
        # charge_violation_ids = np.argwhere(state_transition[:, 2] > np.max(self.states)).squeeze()
        
        # state_transition[discharge_violation_ids, 2] = np.min(self.states)
        # state_transition[charge_violation_ids, 2] = np.max(self.states)
        
        # state_transition[np.concat([charge_violation_ids, discharge_violation_ids]), -1] = np.inf
        
        # time_ids = np.tile(np.array(range(len(self.prices))), reps=[state_transition.shape[0]//len(self.prices),1])
        
        # state_transition_matrix = np.concatenate([time_ids.T.reshape(-1, 1), state_transition], axis=1)
        # return state_transition_matrix
        

In [33]:
np.arange(0, 2)

array([0, 1])

In [34]:
np.array([0, 1]).reshape(1, -1) + np.arange(0, 2).reshape(-1, 1)

array([[0, 1],
       [1, 2]])

In [29]:
prices = np.array([1,5,3,5])
#prices = np.random.uniform(low=1, high=10, size=8760)
states = np.array([0, 1])
actions = np.array([-1, 0, 1])
bs = SimpleBatteryStorageChargesRestricted(states=states, actions=actions, prices=prices, start_state=0, end_state=0, max_charges=1)
bs_matrix = bs.get_state_transitions()
bs_matrix

ValueError: operands could not be broadcast together with shapes (4,3) (2,1) 

In [ ]:
prices = np.array([1,5,3,5])
#prices = np.random.uniform(low=1, high=10, size=8760)
states = np.array([0, 1, 2,3])
actions = np.array([-1, 0, 1])
bs = SimpleBatteryStorage(states=states, actions=actions, prices=prices, start_state=0, end_state=0)
bs_matrix = bs.get_state_transitions()
bs_matrix

array([[ 0.,  0., -1.,  0., inf],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  1.,  1.,  1.],
       [ 0.,  1., -1.,  0., -1.],
       [ 0.,  1.,  0.,  1.,  0.],
       [ 0.,  1.,  1.,  2.,  1.],
       [ 0.,  2., -1.,  1., -1.],
       [ 0.,  2.,  0.,  2.,  0.],
       [ 0.,  2.,  1.,  3.,  1.],
       [ 0.,  3., -1.,  2., -1.],
       [ 0.,  3.,  0.,  3.,  0.],
       [ 0.,  3.,  1.,  3., inf],
       [ 1.,  0., -1.,  0., inf],
       [ 1.,  0.,  0.,  0.,  0.],
       [ 1.,  0.,  1.,  1.,  5.],
       [ 1.,  1., -1.,  0., -5.],
       [ 1.,  1.,  0.,  1.,  0.],
       [ 1.,  1.,  1.,  2.,  5.],
       [ 1.,  2., -1.,  1., -5.],
       [ 1.,  2.,  0.,  2.,  0.],
       [ 1.,  2.,  1.,  3.,  5.],
       [ 1.,  3., -1.,  2., -5.],
       [ 1.,  3.,  0.,  3.,  0.],
       [ 1.,  3.,  1.,  3., inf],
       [ 2.,  0., -1.,  0., inf],
       [ 2.,  0.,  0.,  0.,  0.],
       [ 2.,  0.,  1.,  1.,  3.],
       [ 2.,  1., -1.,  0., -3.],
       [ 2.,  1.,  0.,  1.,  0.],
       [ 2.,  

In [ ]:
# backward
value_list = []
number_state_action_pairs = bs_matrix.shape[0]//len(prices)
for i,t in enumerate(reversed(range(len(prices)))):
    if i == 0:
        previous_state_values = np.zeros(shape=(1, bs.states.shape[-1]))
    else:
        previous_state_values = value_list[i-1]
    
    t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    temp_array = bs_matrix[t_id,...]
    #print(temp_array)
    temp_state_values = previous_state_values[:, np.int32(temp_array[:, -2])].squeeze()

    temp_state_values = temp_state_values + temp_array[:, -1]
    
    value_list.append(np.min(temp_state_values.reshape(bs.states.shape[-1], -1), axis=1).reshape((1, bs.states.shape[-1])))
    
    

In [5]:
value_list

[array([[ 0.        , -4.68328877, -4.68328877, -4.68328877]]),
 array([[  0.        ,  -6.31456373, -10.99785249, -10.99785249]]),
 array([[ -5.14458904,  -9.82787781, -10.99785249, -12.16782717]]),
 array([[ -7.75362169,  -9.82787781, -11.90213393, -13.07210861]]),
 array([[ -7.75362169, -10.68163208, -12.7558882 , -14.83014431]]),
 array([[ -7.75362169, -10.91679796, -13.84480835, -15.91906447]]),
 array([[ -7.75362169, -16.06560901, -19.22878528, -22.15679567]]),
 array([[-13.33039662, -16.49357289, -19.42158327, -22.15679567]]),
 array([[-13.33039662, -18.94508122, -22.10825749, -25.03626787]]),
 array([[-13.33039662, -20.0596354 , -25.67432   , -28.83749627]]),
 array([[-13.95169129, -20.0596354 , -26.16757951, -31.78226411]]),
 array([[-15.24536049, -21.35330461, -26.9679892 , -31.78226411]]),
 array([[-15.24536049, -23.38309728, -29.49104139, -35.10572599]]),
 array([[-16.60522542, -23.38309728, -30.16096914, -36.26891325]]),
 array([[-16.60522542, -26.3411974 , -33.11906926, -

In [6]:
# forward 
forward_value_list = list(reversed(value_list))
chosen_states = []
chosen_actions = []
for t in range(len(prices)):
    t_id = np.argwhere(bs_matrix[:, 0]==t).squeeze()
    temp_array = bs_matrix[t_id,...]
    
    if t == 0:
        if (bs.start_state is not None):
            temp_state = bs.start_state
        
        else:
            temp_state = np.argmin(forward_value_list[t].squeeze())
        chosen_states.append(temp_state)
    else:
        temp_state = chosen_states[t] # not t-1 since we already have the initial state in the chosen states list

    temp_array = temp_array[np.argwhere(temp_array[:, 1]==temp_state).squeeze(), ...]
    
    if t == len(prices)-1:
        next_state_value = np.zeros(shape=(1, bs.states.shape[-1]))
    else:
        next_state_value = forward_value_list[t+1]
    
    costs = temp_array[:, -1]
    diff_arr = (next_state_value[:, np.int32(temp_array[:, -2])] + costs).squeeze()
    
    action_selection = np.argmin(diff_arr).squeeze()
    
    chosen_actions.append(int(temp_array[action_selection, 2]))
    chosen_states.append(int(temp_array[action_selection, 3]))
    
    


In [7]:
chosen_states

[0,
 0,
 1,
 2,
 1,
 2,
 3,
 2,
 1,
 0,
 1,
 0,
 0,
 1,
 2,
 3,
 2,
 1,
 0,
 1,
 2,
 1,
 0,
 1,
 0,
 1,
 2,
 3,
 2,
 3,
 2,
 1,
 0,
 1,
 1,
 0,
 1,
 2,
 1,
 0,
 1,
 2,
 2,
 3,
 2,
 3,
 2,
 3,
 2,
 3,
 2,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 2,
 1,
 0,
 0,
 1,
 1,
 2,
 3,
 2,
 1,
 2,
 3,
 2,
 1,
 2,
 3,
 2,
 1,
 0,
 0,
 1,
 1,
 2,
 3,
 3,
 2,
 1,
 0,
 0,
 1,
 2,
 3,
 2,
 1,
 2,
 3,
 2,
 3,
 2,
 3,
 2,
 3,
 2,
 3,
 2,
 1,
 0,
 1,
 2,
 1,
 2,
 1,
 2,
 3,
 2,
 3,
 2,
 1,
 0,
 0,
 1,
 2,
 3,
 2,
 1,
 2,
 2,
 1,
 0,
 1,
 2,
 1,
 0,
 0,
 0,
 1,
 2,
 3,
 2,
 3,
 2,
 3,
 2,
 1,
 1,
 0,
 1,
 2,
 3,
 2,
 3,
 2,
 3,
 2,
 1,
 0,
 1,
 0,
 1,
 1,
 2,
 1,
 2,
 3,
 2,
 2,
 3,
 2,
 3,
 2,
 3,
 2,
 1,
 0,
 1,
 2,
 1,
 2,
 3,
 2,
 3,
 3,
 2,
 2,
 3,
 2,
 1,
 1,
 2,
 3,
 2,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 2,
 1,
 0,
 1,
 2,
 1,
 1,
 0,
 1,
 0,
 1,
 2,
 3,
 2,
 1,
 2,
 1,
 2,
 1,
 0,
 0,
 0,
 1,
 2,
 3,
 2,
 3,
 2,
 1,
 0,
 1,
 2,
 3,
 2,
 1,
 0,
 1,
 2,
 1,
 0,
 1,
 0,
 1,
 2,
 2,
 3,
 2,
 2,
 3,
 2,
 3,
 2,
 3,
 2,


In [8]:
chosen_actions

[0,
 1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 -1,
 1,
 -1,
 0,
 1,
 1,
 1,
 -1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 1,
 0,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 0,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 0,
 1,
 -1,
 0,
 1,
 1,
 -1,
 -1,
 0,
 1,
 0,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 -1,
 0,
 1,
 0,
 1,
 1,
 0,
 -1,
 -1,
 -1,
 0,
 1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 0,
 1,
 1,
 1,
 -1,
 -1,
 1,
 0,
 -1,
 -1,
 1,
 1,
 -1,
 -1,
 0,
 0,
 1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 0,
 -1,
 1,
 1,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 1,
 -1,
 1,
 0,
 1,
 -1,
 1,
 1,
 -1,
 0,
 1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,
 1,
 1,
 -1,
 1,
 1,
 -1,
 1,
 0,
 -1,
 0,
 1,
 -1,
 -1,
 0,
 1,
 1,
 -1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 1,
 1,
 -1,
 -1,
 1,
 1,
 -1,
 0,
 -1,
 1,
 -1,
 1,
 1,
 1,
 -1,
 -1,
 1,
 -1,
 1,
 -1,
 -1,
 0,
 0,
 1,
 1,
 1,
 -1,
 1,
 -1,
 -1,
 -1,


In [9]:
np.sum(np.array(chosen_actions) * prices)

np.float64(-17729.731026335616)